In [ ]:
import warnings
warnings.filterwarnings("ignore")
import PIL
import os
from scapy.all import rdpcap, IP, UDP, TCP

# Para ver que se ha hecho el paso a .bin correctamente, comparando con Wireshark

In [ ]:
def leer_y_visualizar_bin(ruta_bin):
    try:
        with open(ruta_bin, "rb") as f:
            contenido = f.read()
            hex_str = contenido.hex()

            pares = [hex_str[i:i+2] for i in range(0, len(hex_str), 2)]

            print("Contenido en hexadecimal con espacios:")
            for i in range(0, len(pares), 17):
                linea = " ".join(pares[i:i+17])
                print(linea)

            print("\nContenido en bytes brutos:")
            print(contenido)

            print("\nIntentando decodificar como texto ASCII:")
            print(contenido.decode("ascii", errors="replace"))

    except FileNotFoundError:
        print(f"El archivo no existe: {ruta_bin}")
    except Exception as e:
        print(f"Error al leer el archivo: {e}")

# Meter ruta a archivo .bin para leer
ruta_archivo = r"C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\nontor\bins removals\f_AIM_Chat\f_AIM_Chat_7.bin"

leer_y_visualizar_bin(ruta_archivo)


Contenido en hexadecimal con espacios:
c1 f7 14 66 79 83 07 ce e6 f5 58 33 50 10 13 87 2c
f2 00 00 17 03 02 06 60 84 4b 9c bc 85 56 ad bf 10
75 d1 c4 10 38 96 11 c9 16 70 74 36 dd 1f 75 44 a1
c2 ff ec 50 5a 8c 00 f4 14 e8 bf 89 08 32 88 f4 f9
e7 29 eb 9f 1d 0d d5 d9 96 96 86 dd ac f2 99 e1 e8
d7 72 32 2e 31 ff 9f c9 1c 98 9c 21 4a 04 04 e7 89
72 ca b3 07 81 51 32 f6 c5 bf 97 d9 dc f3 21 b2 40
aa e0 64 8a 4d d0 aa 3e eb fe da a3 b8 bc 11 8c e5
24 16 7d 6b 1f dc 5c 2b ca ab 40 b2 17 b1 1f 93 65
db ab 2b 2d 6a 49 13 6b 04 75 3d f4 75 92 59 71 27
44 19 d9 a3 bc cf 2a e5 2f e5 3f 16 6b 03 de 71 17
13 b3 d1 8f 69 3e 25 7f d1 f1 25 e6 60 98 ee f6 09
78 d5 b2 dc d0 8a 5e 6e fa 38 42 67 51 89 cf d8 6c
69 e7 52 c0 1a 1e e2 37 3b b5 80 33 fe 06 70 a4 3f
0f db e9 44 d0 0b 72 b1 c4 6f de 0d b3 26 05 d7 21
24 e1 9e 8a b4 e5 71 94 29 73 60 83 7e d8 2d 4f be
01 1c 01 98 fa c2 ec d2 de c8 39 a3 69 62 bf 9d 31
27 59 a6 13 a4 d5 97 5c 7b 1c 22 44 b4 f2 e4 1d 83
36 7c 0e 68 76 33 2d b9 56 f7 0f a8 9b 8e 0

## TRANSFORMACION DE .PCAP -> .BIN

In [ ]:
# Parámetros de configuración
HASH_SIZE = 20000 
MAX_PAYLOAD_SIZE = 1024  # Máximo tamaño del paquete a guardar


hash_table = [None] * HASH_SIZE
contador_flujos = 0

# Función para crear un directorio
def create_output_directory(output_dir, flow_id):
    flow_dir = os.path.join(output_dir, str(flow_id))
    os.makedirs(flow_dir, exist_ok=True)
    return flow_dir

# Función para guardar paquetes en archivos
def save_packet_to_file(flow_dir, packet_data, packet_id):
    file_path = os.path.join(flow_dir, f"{packet_id}.bin")
    with open(file_path, 'wb') as file:
        file.write(packet_data[:MAX_PAYLOAD_SIZE])

# Función para analizar un paquete
def analizar_paquete(paquete, output_dir, token, contador):
    if IP not in paquete:
        print("No es un paquete IP. Descartado.")
        return

    payload = bytes(paquete[IP].payload)
    file_path = os.path.join(output_dir, f"{token}_{contador}.bin")
    with open(file_path, 'wb') as file:
        file.write(payload[:MAX_PAYLOAD_SIZE])

# Función principal para procesar múltiples PCAPs y PCAPNGs
def procesar_capturas(input_dir, output_dir):
    global contador_flujos
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(input_dir):
        if not (filename.endswith(".pcap") or filename.endswith(".pcapng")):
            continue

        pcap_path = os.path.join(input_dir, filename)
        if not os.path.isfile(pcap_path):
            continue

        token = os.path.splitext(filename)[0]
        file_output_dir = os.path.join(output_dir, token)
        os.makedirs(file_output_dir, exist_ok=True)

        print(f"Procesando archivo: {filename}...")

        try:
            paquetes = rdpcap(pcap_path)
        except Exception as e:
            print(f"Error al leer {filename}: {e}")
            continue

        contador = 0
        for paquete in paquetes:
            contador += 1
            analizar_paquete(paquete, file_output_dir, token, contador)

        print(f"Archivo procesado. Resultados en: {file_output_dir}")


In [ ]:
## PASAR DE .PCAP A .BIN
INPUT_DIR = r"C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\inputs\nonVpnfiltered\3"
OUTPUT_DIR = r"C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn"

procesar_capturas(INPUT_DIR, OUTPUT_DIR)

Procesando archivo: f_scpDown1.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown1
Procesando archivo: f_scpDown2.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown2
Procesando archivo: f_scpDown3.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown3
Procesando archivo: f_scpDown4.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown4
Procesando archivo: f_scpDown5.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown5
Procesando archivo: f_scpDown6.pcap...
Archivo procesado. Resultados en: C:\Users\migue\Desktop\Cositis\unistuff\TFG\mio\data\outputs\binary_no_vpn\f_scpDown6
Procesando archivo: f_scpUp3.pcap...
Archivo p